# Lesson 9.4: How Do You Handle Multilingual Documents in RAG?

**Companion notebook for Lesson 9.4**

---

| Section | What you will build |
|---|---|
| 1. The Core Problem | Prove English-only embeddings fail on cross-lingual queries — with numbers |
| 2. Strategy A: Pivot Language | Mock translation pipeline + embedding; measure lossy translation |
| 3. Strategy B: Multilingual Model | `paraphrase-multilingual-MiniLM-L12-v2` — one index, all languages |
| 4. Cross-Lingual Retrieval Test | Blog Exercise 2: are same-meaning cross-lingual pairs closer than same-language different-meaning? |
| 5. Language Detection | Character/Unicode heuristic + metadata tagging at index time |
| 6. Language as Metadata | Filter and prefer by language tag; language-aware ranking |
| 7. Per-Language Index Pattern | Separate indexes + language router + multilingual fallback |
| 8. Mixed-Language Chunks | Code-switching examples; how multilingual models handle them |
| 9. Multilingual Synthesis Prompt | Instruct the LLM to answer in the user's language |
| 10. Evaluation | Recall@5 across strategies and languages; visual comparison |
| 11. Claude API | Multilingual synthesis with explicit language instruction |

**Required:** `sentence-transformers`, `numpy`, `matplotlib`  
**No extra installs needed** — language detection is Unicode-based (no fasttext required).  
**Optional (Section 11):** `anthropic`

> **Corpus:** 20 chunks across 5 languages (EN, ES, DE, FR, JA) covering the same four topic areas
> (refund policy, shipping, tier benefits, SLA). This controlled overlap lets us measure cross-lingual
> retrieval quality precisely: each chunk has a known set of equivalents in other languages.


In [ ]:
# Uncomment to install
# !pip install sentence-transformers numpy matplotlib
# !pip install anthropic   # optional — Section 11

%matplotlib inline
import os, re, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict

warnings.filterwarnings('ignore')
os.environ['OMP_NUM_THREADS']         = '1'
os.environ['MKL_NUM_THREADS']         = '1'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '60'
os.environ['TOKENIZERS_PARALLELISM']  = 'false'

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size']      = 11
plt.rcParams['axes.grid']      = True
plt.rcParams['grid.alpha']     = 0.3

def show_plot():
    plt.tight_layout()
    plt.show()

print('Imports ready.')


In [ ]:
# ── Multilingual document corpus ─────────────────────────────────────────────
# 20 chunks across 5 languages, covering 4 topics.
# topic_id links semantically equivalent chunks across languages.

CORPUS = [
    # ── Refund Policy (topic 1) ───────────────────────────────────────────────
    {'id':'en_refund', 'lang':'en', 'topic':1, 'topic_name':'refund_policy',
     'text':(
         'Customers may return any product within 30 days of purchase for a full refund. '
         'Refunds are processed within 5-7 business days. '
         'Software licenses are non-refundable once activated. '
         'Contact returns@techco.com to initiate a return.'
     )},
    {'id':'es_refund', 'lang':'es', 'topic':1, 'topic_name':'refund_policy',
     'text':(
         'Los clientes pueden devolver cualquier producto en un plazo de 30 dias '
         'desde la compra para obtener un reembolso completo. '
         'Los reembolsos se procesan en 5-7 dias habiles. '
         'Las licencias de software no son reembolsables una vez activadas.'
     )},
    {'id':'de_refund', 'lang':'de', 'topic':1, 'topic_name':'refund_policy',
     'text':(
         'Kunden konnen jedes Produkt innerhalb von 30 Tagen nach dem Kauf '
         'gegen vollstandige Ruckerstattung zuruckgeben. '
         'Ruckerstattungen werden innerhalb von 5-7 Werktagen bearbeitet. '
         'Softwarelizenzen sind nach der Aktivierung nicht erstattungsfahig.'
     )},
    {'id':'fr_refund', 'lang':'fr', 'topic':1, 'topic_name':'refund_policy',
     'text':(
         'Les clients peuvent retourner tout produit dans les 30 jours suivant '
         'l\'achat pour un remboursement complet. '
         'Les remboursements sont traites dans un delai de 5 a 7 jours ouvrables. '
         'Les licences logicielles ne sont pas remboursables une fois activees.'
     )},
    {'id':'ja_refund', 'lang':'ja', 'topic':1, 'topic_name':'refund_policy',
     'text':(
         '購入後30日以内であれば、どの製品も全額返金にて返品可能です。'
         '返金処理には5〜7営業日かかります。'
         'ソフトウェアライセンスは一度有効化すると返金対象外となります。'
     )},
    # ── Shipping (topic 2) ───────────────────────────────────────────────────
    {'id':'en_ship', 'lang':'en', 'topic':2, 'topic_name':'shipping',
     'text':(
         'Standard US shipping costs $9.99 and takes 5-7 business days. '
         'Expedited 2-day shipping costs $24.99. '
         'Orders over $150 qualify for free standard shipping. '
         'International shipping rates vary by destination.'
     )},
    {'id':'es_ship', 'lang':'es', 'topic':2, 'topic_name':'shipping',
     'text':(
         'El envio estandar cuesta $9.99 y tarda entre 5 y 7 dias habiles. '
         'El envio urgente de 2 dias cuesta $24.99. '
         'Los pedidos superiores a $150 tienen envio estandar gratuito. '
         'Las tarifas de envio internacional varian segun el destino.'
     )},
    {'id':'de_ship', 'lang':'de', 'topic':2, 'topic_name':'shipping',
     'text':(
         'Der Standardversand kostet $9.99 und dauert 5-7 Werktage. '
         'Der Expressversand innerhalb von 2 Tagen kostet $24.99. '
         'Bestellungen uber $150 erhalten kostenlosen Standardversand. '
         'Internationale Versandkosten variieren je nach Zielort.'
     )},
    {'id':'fr_ship', 'lang':'fr', 'topic':2, 'topic_name':'shipping',
     'text':(
         'La livraison standard aux Etats-Unis coute 9,99 $ et prend 5 a 7 jours ouvrables. '
         'La livraison express en 2 jours coute 24,99 $. '
         'Les commandes superieures a 150 $ beneficient de la livraison standard gratuite.'
     )},
    {'id':'ja_ship', 'lang':'ja', 'topic':2, 'topic_name':'shipping',
     'text':(
         '米国内の通常配送は$9.99で、5〜7営業日かかります。'
         '2日間のエクスプレス配送は$24.99です。'
         '$150以上のご注文は通常配送が無料になります。'
         '国際配送料は配送先によって異なります。'
     )},
    # ── Tier Benefits (topic 3) ───────────────────────────────────────────────
    {'id':'en_tier', 'lang':'en', 'topic':3, 'topic_name':'tier_benefits',
     'text':(
         'Free tier: core features, 2GB storage, community support. '
         'Pro tier ($29/month): unlimited storage, priority email support, API access. '
         'Enterprise tier: dedicated SLA, custom integrations, named account manager. '
         'Upgrades take effect immediately.'
     )},
    {'id':'es_tier', 'lang':'es', 'topic':3, 'topic_name':'tier_benefits',
     'text':(
         'Nivel gratuito: funciones basicas, 2 GB de almacenamiento, soporte comunitario. '
         'Nivel Pro ($29/mes): almacenamiento ilimitado, soporte de correo electrónico prioritario, acceso a API. '
         'Nivel Enterprise: SLA dedicado, integraciones personalizadas, gestor de cuenta nombrado.'
     )},
    {'id':'de_tier', 'lang':'de', 'topic':3, 'topic_name':'tier_benefits',
     'text':(
         'Kostenlose Stufe: Kernfunktionen, 2 GB Speicher, Community-Support. '
         'Pro-Stufe ($29/Monat): unbegrenzter Speicher, priorisierter E-Mail-Support, API-Zugang. '
         'Enterprise-Stufe: dediziertes SLA, individuelle Integrationen, persönlicher Account Manager.'
     )},
    {'id':'fr_tier', 'lang':'fr', 'topic':3, 'topic_name':'tier_benefits',
     'text':(
         'Niveau gratuit: fonctionnalites de base, 2 Go de stockage, support communautaire. '
         'Niveau Pro (29 $/mois): stockage illimite, assistance par email prioritaire, acces API. '
         'Niveau Entreprise: SLA dedie, integrations personnalisees, charge de compte dedié.'
     )},
    {'id':'ja_tier', 'lang':'ja', 'topic':3, 'topic_name':'tier_benefits',
     'text':(
         'フリープラン: 基本機能、2GBストレージ、コミュニティサポート。'
         'プロプラン（月額$29）: 無制限ストレージ、優先メールサポート、APIアクセス。'
         'エンタープライズプラン: 専用SLA、カスタム統合、専任アカウントマネージャー。'
     )},
    # ── SLA / Uptime (topic 4) ───────────────────────────────────────────────
    {'id':'en_sla', 'lang':'en', 'topic':4, 'topic_name':'sla',
     'text':(
         'Pro tier: 99.5% uptime SLA, 24-hour support response time. '
         'Enterprise tier: 99.9% uptime SLA, 4-hour response, named account manager. '
         'Scheduled maintenance windows: Sundays 02:00-04:00 UTC.'
     )},
    {'id':'es_sla', 'lang':'es', 'topic':4, 'topic_name':'sla',
     'text':(
         'Nivel Pro: SLA de disponibilidad del 99.5%, tiempo de respuesta de soporte de 24 horas. '
         'Nivel Enterprise: SLA de disponibilidad del 99.9%, respuesta en 4 horas, gestor dedicado. '
         'Ventanas de mantenimiento: domingos de 02:00 a 04:00 UTC.'
     )},
    {'id':'de_sla', 'lang':'de', 'topic':4, 'topic_name':'sla',
     'text':(
         'Pro-Stufe: 99,5% Betriebszeit-SLA, 24-Stunden-Support-Reaktionszeit. '
         'Enterprise-Stufe: 99,9% Betriebszeit-SLA, 4-Stunden-Reaktionszeit, persönlicher Account Manager. '
         'Wartungsfenster: Sonntags 02:00-04:00 UTC.'
     )},
    {'id':'fr_sla', 'lang':'fr', 'topic':4, 'topic_name':'sla',
     'text':(
         'Niveau Pro: SLA de disponibilite de 99,5%, delai de reponse du support de 24 heures. '
         'Niveau Entreprise: SLA de 99,9%, reponse en 4 heures, responsable de compte dedie. '
         'Fenetres de maintenance: dimanches 02h00-04h00 UTC.'
     )},
    {'id':'ja_sla', 'lang':'ja', 'topic':4, 'topic_name':'sla',
     'text':(
         'プロプラン: 稼働率99.5%のSLA、サポート応答時間24時間。'
         'エンタープライズプラン: 稼働率99.9%のSLA、4時間以内の応答、専任アカウントマネージャー。'
         'メンテナンス時間: 毎週日曜日02:00〜04:00 UTC。'
     )},
]

LANG_NAMES = {'en':'English','es':'Spanish','de':'German','fr':'French','ja':'Japanese'}
LANGS      = list(LANG_NAMES.keys())
TOPICS     = sorted(set(c['topic'] for c in CORPUS))

print(f'Corpus: {len(CORPUS)} chunks  ×  {len(LANGS)} languages  ×  {len(TOPICS)} topics')
for lang in LANGS:
    chunks = [c for c in CORPUS if c['lang']==lang]
    print(f'  {LANG_NAMES[lang]:12s} ({lang}): {len(chunks)} chunks')


In [ ]:
from sentence_transformers import SentenceTransformer, util

print('Loading models (first run downloads ~90MB each)...')

# English-only model (represents Strategy: monolingual index)
en_model = SentenceTransformer('all-MiniLM-L6-v2', device='cpu')
print('  all-MiniLM-L6-v2 loaded  (English-only)')

# Multilingual model (Strategy B)
ml_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2', device='cpu')
print('  paraphrase-multilingual-MiniLM-L12-v2 loaded  (50 languages)')

print()
print('Models ready.')


---
## 1. The Core Problem — English Embeddings Fail Across Languages

An English-only embedding model learned meaning from mostly-English training data.
The word *reembolso* barely appeared next to anything the model knows — so it lands
in a random corner of vector space, far from *refund*, even though they mean the same thing.

```
English index: [refund policy]  [shipping costs]  [SLA uptime]  ...
                    ↑
Query: "política de reembolso" → embedding lands far from any of these
Result: nothing relevant returned
```


In [ ]:
# ── Build English-only index (only the EN chunks) ────────────────────────────
en_chunks  = [c for c in CORPUS if c['lang'] == 'en']
en_embeds  = en_model.encode([c['text'] for c in en_chunks],
                              convert_to_tensor=True, show_progress_bar=False)

def en_retrieve(query, top_k=3):
    q_emb  = en_model.encode(query, convert_to_tensor=True, show_progress_bar=False)
    scores = util.cos_sim(q_emb, en_embeds)[0].cpu().numpy()
    idx    = np.argsort(scores)[::-1][:top_k]
    return [(en_chunks[i], float(scores[i])) for i in idx]


# Cross-lingual queries that should match the same topic
TEST_QUERIES = [
    ('en', 'What is the refund policy?',       1, 'en_refund'),
    ('es', '¿Cuál es la política de reembolso?', 1, 'en_refund'),
    ('de', 'Was ist die Rückerstattungsrichtlinie?', 1, 'en_refund'),
    ('fr', 'Quelle est la politique de remboursement?', 1, 'en_refund'),
    ('ja', '返金ポリシーは何ですか？',           1, 'en_refund'),
    ('en', 'How much does shipping cost?',      2, 'en_ship'),
    ('es', '¿Cuánto cuesta el envío?',          2, 'en_ship'),
    ('de', 'Was kostet der Versand?',           2, 'en_ship'),
]

print('=== English-only index: cross-lingual retrieval test ===\n')
print(f'{"Query (lang)":<50} {"Expected":<12} {"Top-1 match":<12} {"Score":<8} {"Correct?"}')
print('-' * 95)

en_correct, en_total = 0, 0
for lang, query, expected_topic, expected_id in TEST_QUERIES:
    results = en_retrieve(query, top_k=3)
    top_chunk, top_score = results[0]
    correct = top_chunk['id'] == expected_id
    if correct:
        en_correct += 1
    en_total += 1
    mark = 'OK' if correct else 'MISS'
    q_display = f'{query[:40]} ({lang})'
    print(f'{q_display:<50} {expected_id:<12} {top_chunk["id"]:<12} {top_score:.3f}    {mark}')

print()
print(f'English-only model accuracy: {en_correct}/{en_total} = {en_correct/en_total:.0%}')
print()
print('The English-only model gets English queries right.')
print('It fails on non-English queries because "reembolso" and "refund" are far apart in its space.')


---
## 2. Strategy A: Translate Everything to a Pivot Language

Pick English as the pivot. Translate every document to English before indexing.
Translate every query to English before searching.

**Why it works:** your existing English embedding pipeline just works.  
**Why it bites you:** translation is lossy, and errors compound.

```
  Query (DE)  →  translate  →  English query  →  English index  →  English chunk  →  translate  →  Answer (DE)
                                                                                         ↑
                                                              Three places to go wrong
```


In [ ]:
# ── Mock translation system ───────────────────────────────────────────────────
# Simulates a machine translation API.
# In production: use Google Cloud Translation, DeepL, or Helsinki-NLP models.
# We inject a realistic lossiness pattern to demonstrate the pitfall.

TRANSLATIONS_TO_EN = {
    # Spanish → English (good)
    '¿Cuál es la política de reembolso?': 'What is the refund policy?',
    '¿Cuánto cuesta el envío?':            'How much does shipping cost?',
    'Los clientes pueden devolver cualquier producto en un plazo de 30 dias '
    'desde la compra para obtener un reembolso completo. '
    'Los reembolsos se procesan en 5-7 dias habiles. '
    'Las licencias de software no son reembolsables una vez activadas.':
    'Customers can return any product within 30 days of purchase for a full refund. '
    'Refunds are processed in 5-7 business days. '
    'Software licenses are non-refundable once activated.',

    # German → English (with lossiness: Geschäftsführer problem)
    'Was ist die Rückerstattungsrichtlinie?': 'What is the refund policy?',
    'Was kostet der Versand?':                'What does the shipment cost?',  # slightly off
    'Kunden konnen jedes Produkt innerhalb von 30 Tagen nach dem Kauf '
    'gegen vollstandige Ruckerstattung zuruckgeben. '
    'Ruckerstattungen werden innerhalb von 5-7 Werktagen bearbeitet. '
    'Softwarelizenzen sind nach der Aktivierung nicht erstattungsfahig.':
    'Customers can return every product within 30 days after purchase '
    'for a complete reimbursement. '
    'Reimbursements will be processed within 5-7 working days. '
    'Software licenses are not eligible for reimbursement after activation.',
    # Note: "refund" became "reimbursement" — same meaning, different token neighborhood

    # French → English (good)
    'Quelle est la politique de remboursement?': 'What is the refund policy?',

    # Japanese → English (with loss of honorifics and nuance)
    '返金ポリシーは何ですか？':  'What is the refund policy?',
    '購入後30日以内であれば、どの製品も全額返金にて返品可能です。'
    '返金処理には5〜7営業日かかります。'
    'ソフトウェアライセンスは一度有効化すると返金対象外となります。':
    'Products can be returned within 30 days of purchase for a full refund. '
    'Refund processing takes 5-7 business days. '
    'Software licenses cannot be refunded once activated.',
}

def mock_translate_to_en(text):
    """Mock translation to English. Returns (translated, was_cached)."""
    if text in TRANSLATIONS_TO_EN:
        return TRANSLATIONS_TO_EN[text], True
    # For unmapped text: approximate by returning the text as-is
    # (simulates a translation failure / fallback)
    return text, False


# ── Build translated index ─────────────────────────────────────────────────────
translated_chunks = []
for c in CORPUS:
    if c['lang'] == 'en':
        translated_chunks.append({**c, 'translated_text': c['text'], 'loss': False})
    else:
        translated, found = mock_translate_to_en(c['text'])
        translated_chunks.append({**c, 'translated_text': translated, 'loss': not found})

trans_embeds = en_model.encode(
    [c['translated_text'] for c in translated_chunks],
    convert_to_tensor=True, show_progress_bar=False)

def translate_retrieve(query, top_k=3):
    en_query, _ = mock_translate_to_en(query)
    q_emb       = en_model.encode(en_query, convert_to_tensor=True, show_progress_bar=False)
    scores      = util.cos_sim(q_emb, trans_embeds)[0].cpu().numpy()
    idx         = np.argsort(scores)[::-1][:top_k]
    return [(translated_chunks[i], float(scores[i])) for i in idx]


print('=== Translation lossiness demo ===\n')
de_refund_text = [c for c in CORPUS if c['id']=='de_refund'][0]['text']
en_refund_text = [c for c in CORPUS if c['id']=='en_refund'][0]['text']
de_translated, _ = mock_translate_to_en(de_refund_text)

print('Original English:')
print(f'  "{en_refund_text[:120]}"')
print()
print('German translated to English (note: "refund" became "reimbursement"):')
print(f'  "{de_translated[:120]}"')
print()

# Measure cosine similarity between original EN refund and its German translation
en_emb = en_model.encode(en_refund_text, convert_to_tensor=True, show_progress_bar=False)
de_emb = en_model.encode(de_translated,  convert_to_tensor=True, show_progress_bar=False)
sim = float(util.cos_sim(en_emb, de_emb))
print(f'Similarity between English original and translated German: {sim:.3f}')
print('Even after translation, slight wording drift reduces similarity.')
print('Now multiply this across every chunk in a 10,000-chunk index.')

print()
print('=== Strategy A retrieval test ===\n')
print(f'{"Query (lang)":<50} {"Expected":<12} {"Top-1":<12} {"Score":<8} {"Correct?"}')
print('-' * 90)

trans_correct = 0
for lang, query, expected_topic, expected_id in TEST_QUERIES:
    results         = translate_retrieve(query, top_k=3)
    top_chunk, score = results[0]
    correct = top_chunk['topic'] == expected_topic
    if correct:
        trans_correct += 1
    mark = 'OK' if correct else 'MISS'
    print(f'{query[:40]+" ("+lang+")":<50} topic={expected_topic}    '
          f'{top_chunk["id"]:<12} {score:.3f}    {mark}')

print()
print(f'Strategy A (translate pivot) accuracy: {trans_correct}/{len(TEST_QUERIES)} = {trans_correct/len(TEST_QUERIES):.0%}')
print('Better than English-only, but translation lossiness still causes some misses.')


---
## 3. Strategy B: Multilingual Embedding Model

Use a model trained to put semantically equivalent text from different languages
**near each other in vector space**.

Training signal: millions of translation pairs — sentences known to mean the same thing.
The model learns that *refund* and *reembolso* should land in the same neighborhood.

```
  One shared vector space:

  [refund]      ← en  )
  [reembolso]   ← es  )  all close together
  [Rückerstattung] ← de )  in the same cluster
  [返金]        ← ja  )

  [shipping]    ← en  )  different cluster
  [envío]       ← es  )  also close together
```

Index documents in their **original language**. Query in **any language**.


In [ ]:
# Build the multilingual index: ALL 20 chunks in their original language
ml_embeds = ml_model.encode(
    [c['text'] for c in CORPUS],
    convert_to_tensor=True, show_progress_bar=False)

def ml_retrieve(query, top_k=3, lang_filter=None):
    q_emb  = ml_model.encode(query, convert_to_tensor=True, show_progress_bar=False)
    scores = util.cos_sim(q_emb, ml_embeds)[0].cpu().numpy()
    idx    = np.argsort(scores)[::-1]

    results = []
    for i in idx:
        chunk = CORPUS[i]
        if lang_filter and chunk['lang'] not in lang_filter:
            continue
        results.append((chunk, float(scores[i])))
        if len(results) == top_k:
            break
    return results


print('=== Strategy B: Multilingual model retrieval test ===\n')
print(f'{"Query (lang)":<50} {"Expected topic":<16} {"Top-1":<12} {"Score":<8} {"Correct?"}')
print('-' * 95)

ml_correct = 0
for lang, query, expected_topic, expected_id in TEST_QUERIES:
    results           = ml_retrieve(query, top_k=3)
    top_chunk, score  = results[0]
    correct           = top_chunk['topic'] == expected_topic
    if correct:
        ml_correct += 1
    mark = 'OK' if correct else 'MISS'
    print(f'{query[:40]+" ("+lang+")":<50} topic={expected_topic}          '
          f'{top_chunk["id"]:<12} {score:.3f}    {mark}')

print()
print(f'Strategy B (multilingual model) accuracy: {ml_correct}/{len(TEST_QUERIES)} = {ml_correct/len(TEST_QUERIES):.0%}')
print()
print('Summary:')
print(f'  English-only (monolingual): {en_correct}/{len(TEST_QUERIES)} = {en_correct/len(TEST_QUERIES):.0%}')
print(f'  Strategy A (translate):     {trans_correct}/{len(TEST_QUERIES)} = {trans_correct/len(TEST_QUERIES):.0%}')
print(f'  Strategy B (multilingual):  {ml_correct}/{len(TEST_QUERIES)} = {ml_correct/len(TEST_QUERIES):.0%}')


---
## 4. Cross-Lingual Retrieval Test — Blog Exercise 2

The key question for any multilingual model:

> Are two sentences with the **same meaning in different languages** closer to each other
> than two sentences with **different meanings in the same language**?

If yes: the model has learned real cross-lingual meaning.  
If no: it's just matching language tokens, not semantics.

```
  sim("refund policy EN", "política de reembolso ES")  vs.
  sim("refund policy EN", "how do I bake a cake EN")
  Should be: first pair > second pair
```


In [ ]:
# Blog Exercise 2: cross-lingual similarity test
CROSS_LINGUAL_PAIRS = [
    # (text_a, text_b, lang_a, lang_b, relationship)
    ('What is the refund policy?',
     '¿Cuál es la política de reembolso?',
     'en', 'es', 'same-meaning cross-lingual'),
    ('What is the refund policy?',
     'Was ist die Rückerstattungsrichtlinie?',
     'en', 'de', 'same-meaning cross-lingual'),
    ('What is the refund policy?',
     '返金ポリシーは何ですか？',
     'en', 'ja', 'same-meaning cross-lingual'),
    ('What is the refund policy?',
     'How do I bake a cake?',
     'en', 'en', 'different-meaning same-language'),
    ('What is the refund policy?',
     'What time is it in Tokyo?',
     'en', 'en', 'different-meaning same-language'),
    ('¿Cuál es la política de reembolso?',
     'Was ist die Rückerstattungsrichtlinie?',
     'es', 'de', 'same-meaning cross-lingual (no EN)'),
]

def compare_sims(model, label):
    print(f'=== {label} ===')
    cross_sims  = []
    within_sims = []
    for a, b, la, lb, rel in CROSS_LINGUAL_PAIRS:
        ea = model.encode(a, convert_to_tensor=True, show_progress_bar=False)
        eb = model.encode(b, convert_to_tensor=True, show_progress_bar=False)
        sim = float(util.cos_sim(ea, eb))
        category = 'cross-lingual' if la != lb or 'cross' in rel else 'same-language'
        if 'same-meaning cross' in rel:
            cross_sims.append(sim)
        else:
            within_sims.append(sim)
        print(f'  [{la}/{lb}] {rel:<35} sim={sim:.3f}')
    print(f'  Avg same-meaning cross-lingual : {np.mean(cross_sims):.3f}')
    print(f'  Avg different-meaning same-lang: {np.mean(within_sims):.3f}')
    verdict = 'PASS' if np.mean(cross_sims) > np.mean(within_sims) else 'FAIL'
    print(f'  Cross-lingual > within-language? {verdict}')
    print()
    return cross_sims, within_sims


en_cross, en_within = compare_sims(en_model, 'English-only model (all-MiniLM-L6-v2)')
ml_cross, ml_within = compare_sims(ml_model, 'Multilingual model (paraphrase-multilingual-MiniLM-L12-v2)')


In [ ]:
# Visualise: how models arrange cross-lingual vs. same-meaning pairs
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pairs_en  = []
pairs_ml  = []
pair_types = []

for a, b, la, lb, rel in CROSS_LINGUAL_PAIRS:
    ea_en = en_model.encode(a, convert_to_tensor=True, show_progress_bar=False)
    eb_en = en_model.encode(b, convert_to_tensor=True, show_progress_bar=False)
    ea_ml = ml_model.encode(a, convert_to_tensor=True, show_progress_bar=False)
    eb_ml = ml_model.encode(b, convert_to_tensor=True, show_progress_bar=False)
    pairs_en.append(float(util.cos_sim(ea_en, eb_en)))
    pairs_ml.append(float(util.cos_sim(ea_ml, eb_ml)))
    pair_types.append('same-meaning\ncross-lingual' if 'cross' in rel else 'diff-meaning\nsame-lang')

x     = np.arange(len(CROSS_LINGUAL_PAIRS))
w     = 0.35
colors_type = ['#1565C0' if 'cross' in t else '#E53935' for t in pair_types]

for ax, scores, model_name in [
    (axes[0], pairs_en, 'English-only (all-MiniLM-L6-v2)'),
    (axes[1], pairs_ml, 'Multilingual (paraphrase-multilingual-MiniLM-L12-v2)'),
]:
    bars = ax.bar(x, scores, color=colors_type, alpha=0.85, width=0.6)
    ax.set_xticks(x)
    labs = [f'{a[:20]}\nvs.\n{b[:20]}' for a, b, *_ in CROSS_LINGUAL_PAIRS]
    ax.set_xticklabels([f'Pair {i+1}' for i in range(len(CROSS_LINGUAL_PAIRS))], fontsize=9)
    ax.set_ylim(-0.1, 1.1)
    ax.set_ylabel('Cosine similarity')
    ax.set_title(model_name, fontweight='bold', fontsize=10)
    ax.axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='similarity=0.5')
    for bar, val in zip(bars, scores):
        ax.text(bar.get_x()+bar.get_width()/2, val+0.02,
                f'{val:.2f}', ha='center', fontsize=8)

legend_handles = [
    mpatches.Patch(color='#1565C0', label='Same-meaning, cross-lingual'),
    mpatches.Patch(color='#E53935', label='Different-meaning, same-language'),
]
fig.legend(handles=legend_handles, loc='lower center', ncol=2, fontsize=10,
           bbox_to_anchor=(0.5, -0.05))
fig.suptitle('Cross-Lingual Similarity Test\n'
             'Blue bars (cross-lingual, same meaning) should be ABOVE red bars (same-lang, diff meaning)',
             fontsize=11, fontweight='bold')
show_plot()

print('Interpretation:')
print('  English-only model: cross-lingual pairs are NOT reliably above same-lang different-meaning.')
print('  Multilingual model: cross-lingual same-meaning pairs score higher — real cross-lingual understanding.')


---
## 5. Language Detection — Tag Every Chunk at Index Time

Even with a multilingual model, you need language tags on every chunk.
They let you filter, prefer, route, and audit later — none of this is possible without them.

> Tag now; decide policies later.

In production: use **fasttext `lid.176`** (176 languages, ~1ms, ~100MB model) or
**`langdetect`** (Python, no extra download). Here we use a Unicode/character-based
heuristic — no extra install needed, covers the five demo languages.

> **Don't detect on tiny chunks (< 50 chars).** Detection accuracy drops sharply
> on short text. Detect at the document level and propagate to chunks.


In [ ]:
def detect_language(text):
    """
    Unicode-range heuristic language detector.
    Production alternative: fasttext lid.176 or langdetect.

    Returns (lang_code, confidence).
    Covers: ja, zh, ko, ar, de, fr, es, ru, en.
    """
    if len(text.strip()) < 10:
        return 'unknown', 0.0

    chars = text
    n = max(len(chars), 1)

    # CJK / Japanese
    ja_chars = sum(1 for c in chars if
                   (0x3040 <= ord(c) <= 0x30FF) or  # hiragana/katakana
                   (0x4E00 <= ord(c) <= 0x9FFF))    # kanji
    if ja_chars / n > 0.10:
        return 'ja', min(0.99, ja_chars / n * 3)

    # Arabic
    ar_chars = sum(1 for c in chars if 0x0600 <= ord(c) <= 0x06FF)
    if ar_chars / n > 0.10:
        return 'ar', min(0.99, ar_chars / n * 3)

    # Cyrillic (Russian etc.)
    cy_chars = sum(1 for c in chars if 0x0400 <= ord(c) <= 0x04FF)
    if cy_chars / n > 0.10:
        return 'ru', min(0.99, cy_chars / n * 3)

    # Latin-script languages: use common function words
    text_lower = text.lower()
    words = re.findall(r'[a-zA-Zäöüßàâéèêëîïôùûçñãõ]+', text_lower)

    DE_WORDS = {'die','der','und','ist','von','des','im','zu','den','das','eine','nicht',
                'mit','bei','nach','aus','auch','werden','kunden','tagen','stufe'}
    FR_WORDS = {'les','des','est','une','dans','pour','sur','avec','par','qui','que','de',
                'la','le','en','ou','remboursement','clients','livraison','niveau'}
    ES_WORDS = {'los','las','del','que','una','por','con','para','como','es','en','se',
                'reembolso','envio','clientes','dias'}

    word_set = set(words)
    de_score = len(word_set & DE_WORDS)
    fr_score = len(word_set & FR_WORDS)
    es_score = len(word_set & ES_WORDS)

    scores = {'de': de_score, 'fr': fr_score, 'es': es_score}
    best = max(scores, key=scores.get)
    if scores[best] >= 2:
        conf = min(0.95, scores[best] * 0.15)
        return best, conf

    return 'en', 0.80


# ── Test on corpus chunks ─────────────────────────────────────────────────────
print(f'{"Chunk ID":<15} {"True lang":<12} {"Detected":<12} {"Confidence":<12} {"Correct?"}')
print('-' * 65)
correct = 0
for c in CORPUS:
    detected, conf = detect_language(c['text'])
    ok = detected == c['lang']
    if ok:
        correct += 1
    mark = 'OK' if ok else f'! (got {detected})'
    print(f'{c["id"]:<15} {c["lang"]:<12} {detected:<12} {conf:<12.2f} {mark}')

print(f'\nDetection accuracy: {correct}/{len(CORPUS)} = {correct/len(CORPUS):.0%}')
print()
print('Production tip: use fasttext lid.176 for 176-language support at < 2ms per chunk.')
print('Detect at document level; propagate to all child chunks.')


In [ ]:
# ── Tag every chunk at index time ───────────────────────────────────────────
INDEXED_CHUNKS = []
for c in CORPUS:
    detected_lang, conf = detect_language(c['text'])
    INDEXED_CHUNKS.append({
        **c,
        'detected_lang':       detected_lang,
        'lang_confidence':     round(conf, 3),
        'char_count':          len(c['text']),
    })

# Encode all chunks once
ml_all_embeds = ml_model.encode(
    [c['text'] for c in INDEXED_CHUNKS],
    convert_to_tensor=True, show_progress_bar=False)


def ml_retrieve_filtered(query, lang_filter=None, prefer_lang=None, top_k=5):
    """
    Multilingual retrieval with language metadata:
    - lang_filter: only return chunks in these languages
    - prefer_lang: boost chunks in this language (move them up in ranking)
    """
    q_emb  = ml_model.encode(query, convert_to_tensor=True, show_progress_bar=False)
    scores = util.cos_sim(q_emb, ml_all_embeds)[0].cpu().numpy()

    results = []
    for i, (chunk, score) in enumerate(zip(INDEXED_CHUNKS, scores)):
        if lang_filter and chunk['lang'] not in lang_filter:
            continue
        boosted = score * 1.05 if prefer_lang and chunk['lang'] == prefer_lang else score
        results.append((chunk, float(score), boosted))

    results.sort(key=lambda x: x[2], reverse=True)
    return [(c, s) for c, s, _ in results[:top_k]]


print('=== Language metadata filtering demo ===\n')
q = '¿Cuál es la política de reembolso?'

print(f'Query: "{q}"\n')

print('A. No filter (all 20 chunks eligible):')
for c, s in ml_retrieve_filtered(q, top_k=3):
    print(f'  [{c["lang"]}] {c["id"]:<15} score={s:.3f}  {c["text"][:60]}...')

print()
print('B. Filter: only Spanish results:')
for c, s in ml_retrieve_filtered(q, lang_filter=['es'], top_k=3):
    print(f'  [{c["lang"]}] {c["id"]:<15} score={s:.3f}  {c["text"][:60]}...')

print()
print('C. Prefer Spanish, but include English as fallback:')
for c, s in ml_retrieve_filtered(q, lang_filter=['es','en'], prefer_lang='es', top_k=3):
    print(f'  [{c["lang"]}] {c["id"]:<15} score={s:.3f}  {c["text"][:60]}...')

print()
print('The metadata tag makes all three retrieval strategies possible.')
print('Without it, you cannot filter, prefer, or audit by language after indexing.')


---
## 7. Per-Language Index Pattern

Alternative architecture: keep **separate vector stores per language**.
A language router sends each query to the right store.

```
  Query (DE)  →  Language router  →  German index  →  German chunks
                      ↓ if German index returns nothing
               Multilingual fallback index  →  any language
```

**When this wins:** you have lots of content per language; each language can use its best
monolingual model; users stay within one language per session.  
**When it fails:** sparse languages (German user asks something only in English docs);
code-switching queries; multilingual session patterns.

**Practical compromise:** per-language primary indexes + one multilingual fallback.


In [ ]:
# ── Build per-language indexes ───────────────────────────────────────────────
LANG_INDEXES = {}
for lang in LANGS:
    lang_chunks = [c for c in CORPUS if c['lang'] == lang]
    if not lang_chunks:
        continue
    # Each language index uses the MULTILINGUAL model here.
    # In production: use the best monolingual model for each language.
    embeds = ml_model.encode(
        [c['text'] for c in lang_chunks],
        convert_to_tensor=True, show_progress_bar=False)
    LANG_INDEXES[lang] = {'chunks': lang_chunks, 'embeds': embeds}

# Multilingual fallback uses all chunks
LANG_INDEXES['fallback'] = {'chunks': CORPUS,
                             'embeds': ml_all_embeds}

print('Per-language indexes:')
for lang, idx in LANG_INDEXES.items():
    name = LANG_NAMES.get(lang, 'Multilingual fallback')
    print(f'  {lang:<12} {name:<22} {len(idx["chunks"])} chunks')


def per_lang_retrieve(query, query_lang=None, top_k=3,
                      min_score_threshold=0.35):
    """
    1. If query_lang known, search that language index first.
    2. If results are weak (max score < threshold), fall back to multilingual index.
    """
    if query_lang and query_lang in LANG_INDEXES:
        idx    = LANG_INDEXES[query_lang]
        q_emb  = ml_model.encode(query, convert_to_tensor=True, show_progress_bar=False)
        scores = util.cos_sim(q_emb, idx['embeds'])[0].cpu().numpy()
        top_idx = np.argsort(scores)[::-1][:top_k]
        results = [(idx['chunks'][i], float(scores[i])) for i in top_idx]
        max_score = results[0][1] if results else 0.0

        if max_score >= min_score_threshold:
            return results, f'{query_lang}-index'

        # Weak results: fall back
        print(f'  [Router] {query_lang} index returned weak results (max={max_score:.3f}), '
              f'falling back to multilingual index')

    # Fallback: multilingual index
    idx    = LANG_INDEXES['fallback']
    q_emb  = ml_model.encode(query, convert_to_tensor=True, show_progress_bar=False)
    scores = util.cos_sim(q_emb, idx['embeds'])[0].cpu().numpy()
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [(idx['chunks'][i], float(scores[i])) for i in top_idx], 'multilingual-fallback'


print()
print('=== Per-language index routing demo ===\n')
routing_demos = [
    ('Was ist die Rückerstattungsrichtlinie?', 'de', 'German question → German index'),
    ('¿Cuál es la política de reembolso?',     'es', 'Spanish question → Spanish index'),
    ('Wie viel kostet der Versand?',            'de', 'German shipping question'),
    ('What is the refund policy?',             'en', 'English → English index'),
]
for q, lang, note in routing_demos:
    results, source = per_lang_retrieve(q, query_lang=lang, top_k=2)
    top, score = results[0]
    print(f'{note}')
    print(f'  Query  : "{q}" | Source index: {source}')
    print(f'  Top-1  : [{top["lang"]}] {top["id"]}  score={score:.3f}')
    print()


---
## 8. Mixed-Language Chunks — Code-Switching

Real documents mix languages. A German technical doc quotes English error messages.
A French policy references an English law. A Japanese support ticket uses English product names.

```
"Wenn der Server einen 'Connection refused' Fehler zurückgibt,
 prüfen Sie die firewall settings im /etc/hosts file."
```

What language is that? **Both.** The term for this is *code-switching*.

**Multilingual models handle it gracefully** — they've seen code-switching during training.
Translation-based approaches choke: the translator may or may not translate
"firewall settings", and may decide differently each time.

How to tag mixed chunks: use the **primary language** and set a `mixed=True` flag.


In [ ]:
MIXED_CHUNKS = [
    {'id':'mixed_de_en', 'primary_lang':'de', 'mixed':True,
     'text': (
         "Wenn der Server einen 'Connection refused' Fehler zurückgibt, "
         "prüfen Sie die firewall settings im /etc/hosts file. "
         "Der error code 403 bedeutet 'Access Denied'."
     )},
    {'id':'mixed_ja_en', 'primary_lang':'ja', 'mixed':True,
     'text': (
         'APIキーは「Settings > API Keys」から取得できます。'
         '"Invalid API Key"エラーが発生した場合は、'
         'キーが正しくコピーされているか確認してください。'
     )},
    {'id':'mixed_fr_en', 'primary_lang':'fr', 'mixed':True,
     'text': (
         "En cas d'erreur 'Rate Limit Exceeded', attendez 60 secondes "
         "avant de retry. Le dashboard affiche votre current usage."
     )},
]

# Queries in different languages that should match these mixed-language chunks
mixed_queries = [
    ('Connection refused error in the server', 'en',
     'mixed_de_en', 'English query → German/English chunk'),
    ('Wie behebe ich den Connection refused Fehler?', 'de',
     'mixed_de_en', 'German query → same German/English chunk'),
    ('Where do I find my API key?', 'en',
     'mixed_ja_en', 'English query → Japanese/English chunk'),
    ('Rate limit error how to fix', 'en',
     'mixed_fr_en', 'English query → French/English chunk'),
]

# Embed mixed chunks
mixed_embeds = ml_model.encode(
    [c['text'] for c in MIXED_CHUNKS],
    convert_to_tensor=True, show_progress_bar=False)

print('=== Code-switching retrieval: multilingual model ===\n')
print(f'{"Query":<50} {"Expected":<14} {"Top match":<14} {"Score"}')
print('-' * 88)
correct = 0
for q, q_lang, expected_id, note in mixed_queries:
    q_emb  = ml_model.encode(q, convert_to_tensor=True, show_progress_bar=False)
    scores = util.cos_sim(q_emb, mixed_embeds)[0].cpu().numpy()
    best   = int(np.argmax(scores))
    top    = MIXED_CHUNKS[best]
    score  = float(scores[best])
    ok     = top['id'] == expected_id
    if ok:
        correct += 1
    mark = 'OK' if ok else '!'
    print(f'{q[:48]:<50} {expected_id:<14} {top["id"]:<14} {score:.3f} {mark}')

print(f'\nCode-switching retrieval: {correct}/{len(mixed_queries)} correct')
print()
print('The multilingual model handles code-switching because:')
print('  1. It was trained on text where languages mix (technical docs, social media).')
print('  2. English technical terms ("connection refused", "API key") appear in all')
print('     languages during training, so they share vector space across all of them.')
print()

# Show language detection on mixed chunks
print('Language detection on mixed chunks (note: detects PRIMARY language):')
for c in MIXED_CHUNKS:
    detected, conf = detect_language(c['text'])
    print(f'  [{c["id"]}] primary={c["primary_lang"]}  '
          f'detected={detected} (conf={conf:.2f})')


---
## 9. Multilingual Synthesis Prompt

Modern LLMs are natively multilingual. The hard part isn't generation — it's
instructing them clearly.

**Without explicit instruction:** the LLM defaults to English when most of the prompt
context (system instructions + retrieved chunks) is in English, even if the user asked in Japanese.

**Fix:** detect the user's language explicitly, then pass it in the prompt.
Never make the LLM guess which language to respond in.

```python
SYNTHESIS_PROMPT = '''
Answer the user's question using ONLY the retrieved context.

LANGUAGE RULES:
1. Answer in EXACTLY this language: {lang_name}
2. Do not switch to English even if the retrieved chunks are in English.
3. If you quote a source phrase, keep the original and add a translation in parentheses.
4. If context is insufficient, say so in {lang_name}.
   NEVER fabricate an answer.

User question: {question}
Retrieved context: {chunks}
Answer ({lang_name}):
'''
```


In [ ]:
SYNTHESIS_PROMPT = """\
You are a helpful support assistant. Answer the user's question using
ONLY the retrieved context below.

LANGUAGE RULES:
1. Answer in EXACTLY this language: {lang_name}
2. Do not switch to English even if the retrieved chunks are in English.
3. If you quote a source phrase, keep the original and add a translation in parentheses.
4. If the retrieved context is insufficient, say so in {lang_name}.
   Do NOT fabricate information.

Detected user language: {lang_code} ({lang_name})
User question: {question}

Retrieved context:
{chunks}

Answer ({lang_name}):"""


class MockSynthesisLLM:
    """
    Simulates a multilingual LLM.
    Uses retrieved context to construct a response in the correct language.
    Replace with Claude API in Section 11.
    """

    ANSWER_TEMPLATES = {
        'en': lambda ctx: f'Based on our policy: {ctx[:200]}',
        'es': lambda ctx: f'Según nuestra política: {ctx[:200]}',
        'de': lambda ctx: f'Gemäß unserer Richtlinie: {ctx[:200]}',
        'fr': lambda ctx: f'Selon notre politique: {ctx[:200]}',
        'ja': lambda ctx: f'当社のポリシーによると: {ctx[:200]}',
    }

    def synthesize(self, question, lang_code, chunks):
        ctx      = ' '.join(c['text'] for c in chunks)
        template = self.ANSWER_TEMPLATES.get(lang_code,
                                             self.ANSWER_TEMPLATES['en'])
        return template(ctx)


llm = MockSynthesisLLM()


def multilingual_rag(question):
    # 1. Detect language
    q_lang, q_conf = detect_language(question)

    # 2. Retrieve (multilingual model, prefer user language)
    results = ml_retrieve_filtered(
        question,
        lang_filter=[q_lang, 'en'],   # prefer user lang + English fallback
        prefer_lang=q_lang,
        top_k=3)

    chunks = [c for c, _ in results]

    # 3. Synthesise in user's language
    lang_name = LANG_NAMES.get(q_lang, q_lang)
    answer    = llm.synthesize(question, q_lang, chunks)

    return {
        'question':    question,
        'detected_lang': q_lang,
        'lang_name':   lang_name,
        'chunks':      [c['id'] for c in chunks],
        'answer':      answer,
    }


print('=== Multilingual RAG pipeline with language-aware synthesis ===\n')
synthesis_demos = [
    'What is the refund policy?',
    '¿Cuál es la política de reembolso?',
    'Was ist die Rückerstattungsrichtlinie?',
    'Quelle est la politique de remboursement?',
    '返金ポリシーは何ですか？',
]
for q in synthesis_demos:
    result = multilingual_rag(q)
    print(f'Q [{result["detected_lang"]}]: {q}')
    print(f'  Chunks retrieved: {result["chunks"]}')
    print(f'  Answer ({result["lang_name"]}): {result["answer"][:120]}...')
    print()

print('Key: the synthesis instruction explicitly names the response language.')
print('Without "Answer in {lang_name}", Claude/GPT default to English when')
print('the retrieved chunks are in English — even when the user asked in Japanese.')


---
## 10. Evaluation — Recall@5 Across Strategies and Languages

Recall@K: did the correct chunk appear in the top-K results?

We test 4 queries × 5 languages = 20 total queries.  
For each query, the "correct" result is the chunk with the same topic in ANY language.


In [ ]:
# Full evaluation: all 5 query languages × all 4 topics
EVAL_QUERIES = [
    ('en', 'What is the refund policy?',                  1),
    ('en', 'How much does shipping cost?',                2),
    ('en', 'What are the differences between tiers?',     3),
    ('en', 'What is the uptime SLA for enterprise?',      4),
    ('es', '¿Cuál es la política de reembolso?',          1),
    ('es', '¿Cuánto cuesta el envío?',                   2),
    ('es', '¿Cuáles son las diferencias entre niveles?',  3),
    ('es', '¿Qué SLA de disponibilidad ofrece Enterprise?', 4),
    ('de', 'Was ist die Rückerstattungsrichtlinie?',      1),
    ('de', 'Was kostet der Versand?',                     2),
    ('de', 'Welche Unterschiede gibt es zwischen den Stufen?', 3),
    ('de', 'Was ist das Uptime-SLA für Enterprise?',      4),
    ('fr', 'Quelle est la politique de remboursement?',   1),
    ('fr', 'Combien coûte la livraison?',                 2),
    ('fr', 'Quelles sont les différences entre les niveaux?', 3),
    ('fr', 'Quel est le SLA de disponibilité Enterprise?', 4),
    ('ja', '返金ポリシーは何ですか？',                    1),
    ('ja', '送料はいくらですか？',                        2),
    ('ja', '各プランの違いは何ですか？',                   3),
    ('ja', 'エンタープライズの稼働率SLAは？',              4),
]

K = 5

def recall_at_k(retrieve_fn, queries, k):
    """Compute recall@k: correct topic in top-k returned chunks."""
    per_lang = defaultdict(list)
    overall  = []
    for lang, q, expected_topic in queries:
        results = retrieve_fn(q)
        top_k_topics = [c['topic'] for c, _ in results[:k]]
        hit = expected_topic in top_k_topics
        per_lang[lang].append(int(hit))
        overall.append(int(hit))
    return per_lang, np.mean(overall)

# ── English-only model (can only index English chunks) ───────────────────────
def en_only_retrieve(q):
    q_emb  = en_model.encode(q, convert_to_tensor=True, show_progress_bar=False)
    scores = util.cos_sim(q_emb, en_embeds)[0].cpu().numpy()
    idx    = np.argsort(scores)[::-1][:K]
    return [(en_chunks[i], float(scores[i])) for i in idx]

# ── Pivot translation strategy ────────────────────────────────────────────────
def pivot_retrieve(q):
    return translate_retrieve(q, top_k=K)

# ── Multilingual model ────────────────────────────────────────────────────────
def ml_only_retrieve(q):
    return ml_retrieve_filtered(q, top_k=K)

# ── Per-language + fallback ────────────────────────────────────────────────────
def per_lang_retrieve_fn(q):
    lang, _ = detect_language(q)
    results, _ = per_lang_retrieve(q, query_lang=lang, top_k=K)
    return results


strategies = [
    ('English-only',        en_only_retrieve),
    ('Strategy A (pivot)',  pivot_retrieve),
    ('Strategy B (ml model)', ml_only_retrieve),
    ('Per-lang + fallback', per_lang_retrieve_fn),
]

results_all = {}
print(f'{"Strategy":<28} {"EN":<8} {"ES":<8} {"DE":<8} {"FR":<8} {"JA":<8} {"Overall"}')
print('-' * 80)
for name, fn in strategies:
    per_lang, overall = recall_at_k(fn, EVAL_QUERIES, K)
    results_all[name] = {'per_lang': per_lang, 'overall': overall}
    lang_scores = ' '.join(f'{np.mean(per_lang[l]):.0%}    ' for l in LANGS)
    print(f'{name:<28} {lang_scores}{overall:.0%}')

print(f'\nRecall@{K} — higher is better.  {len(EVAL_QUERIES)} total queries.')
print('Non-English scores on English-only model are low — confirms the core problem.')
print('Multilingual model recovers recall across all languages.')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

strategy_names = [n for n, _ in strategies]
colors = ['#E53935', '#F57F17', '#1565C0', '#2E7D32']

# Left: per-language recall for each strategy
x = np.arange(len(LANGS))
w = 0.20
for i, (name, _) in enumerate(strategies):
    per_lang = results_all[name]['per_lang']
    vals = [np.mean(per_lang[l]) for l in LANGS]
    axes[0].bar(x + i*w - 1.5*w, vals, w,
                color=colors[i], alpha=0.85, label=name)

axes[0].set_xticks(x)
axes[0].set_xticklabels([LANG_NAMES[l] for l in LANGS])
axes[0].set_ylim(0, 1.2)
axes[0].set_ylabel(f'Recall@{K}')
axes[0].set_title(f'Per-Language Recall@{K}', fontweight='bold')
axes[0].legend(fontsize=8, loc='lower right')

# Right: overall recall
overalls = [results_all[n]['overall'] for n, _ in strategies]
bars = axes[1].bar(range(len(strategies)), overalls, color=colors, alpha=0.85, width=0.6)
axes[1].set_xticks(range(len(strategies)))
axes[1].set_xticklabels([n.replace(' (',  '\n(') for n, _ in strategies], fontsize=9)
axes[1].set_ylim(0, 1.2)
axes[1].set_ylabel(f'Overall Recall@{K}')
axes[1].set_title('Overall Recall@5 by Strategy', fontweight='bold')
for bar, val in zip(bars, overalls):
    axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.02,
                 f'{val:.0%}', ha='center', fontsize=13, fontweight='bold')

show_plot()
print('Key finding: English-only model collapses for non-English queries.')
print('Multilingual model restores recall to near-English levels across all languages.')


---
## 11. Claude API — Multilingual Synthesis

Claude is natively multilingual. The key: **always pass the detected language explicitly**.

```python
# WITHOUT explicit language instruction:
# Claude sees English system prompt + English chunks → defaults to English reply
# even when the user asked in Japanese.

# WITH explicit language instruction:
# "Answer in Japanese (ja). Do not switch to English."
# → Claude answers in Japanese regardless of chunk language.
```

Install: `pip install anthropic`  
Set key: `export ANTHROPIC_API_KEY=sk-ant-...`


In [ ]:
ANTHROPIC_AVAILABLE = False
try:
    import anthropic
    ANTHROPIC_AVAILABLE = bool(os.environ.get('ANTHROPIC_API_KEY'))
except ImportError:
    pass

ML_SYNTHESIS_PROMPT = """\
You are a helpful support assistant. Answer the user's question using ONLY
the retrieved context below.

CRITICAL LANGUAGE RULES:
1. The user's language is: {lang_name} (code: {lang_code})
2. Answer in {lang_name}. Do not switch to English under any circumstances.
3. If you need to quote a phrase from an English source, keep the English
   and add a translation in parentheses immediately after.
4. If the retrieved context is insufficient to answer, say so in {lang_name}.
   Do NOT fabricate any information.

User question: {question}

Retrieved context (may be in any language — translate the meaning, not just words):
{chunks}

Answer in {lang_name}:"""


if ANTHROPIC_AVAILABLE:
    client = anthropic.Anthropic()

    def claude_multilingual_rag(question):
        # Step 1: detect language
        q_lang, q_conf = detect_language(question)
        lang_name      = LANG_NAMES.get(q_lang, q_lang)

        # Step 2: retrieve with multilingual model
        results = ml_retrieve_filtered(
            question, lang_filter=[q_lang, 'en'], prefer_lang=q_lang, top_k=3)
        chunks  = [c for c, _ in results]

        chunk_text = '\n---\n'.join(
            f'[{c["lang"]}] {c["text"]}' for c in chunks)

        # Step 3: synthesise in user's language
        prompt = ML_SYNTHESIS_PROMPT.format(
            lang_code=q_lang,
            lang_name=lang_name,
            question=question,
            chunks=chunk_text,
        )

        resp = client.messages.create(
            model='claude-sonnet-4-6',
            max_tokens=600,
            messages=[{'role': 'user', 'content': prompt}],
        )
        answer = resp.content[0].text.strip()

        print(f'[{q_lang}] Q: {question}')
        print(f'  Detected: {lang_name} (conf={q_conf:.2f})')
        print(f'  Chunks: {[c["id"] for c in chunks]}')
        print(f'  Answer: {answer[:300]}')
        print()
        return answer


    claude_queries = [
        'What is the refund policy?',
        '¿Cuál es la política de reembolso?',
        'Was ist die Rückerstattungsrichtlinie?',
        '返金ポリシーは何ですか？',
    ]
    print('=== Claude Multilingual RAG ===\n')
    for q in claude_queries:
        claude_multilingual_rag(q)

else:
    print('anthropic not installed or ANTHROPIC_API_KEY not set.')
    print()
    print('To enable:')
    print('  pip install anthropic')
    print('  export ANTHROPIC_API_KEY=sk-ant-...')
    print()
    print('Two-part prompt design in ML_SYNTHESIS_PROMPT:')
    print('  Part 1: "The user\'s language is: {lang_name} (code: {lang_code})"')
    print('          This makes the language explicit — Claude cannot miss it.')
    print('  Part 2: "Answer in {lang_name}. Do not switch to English."')
    print('          The "do not switch" clause overrides the English bias from')
    print('          English retrieved chunks and English system instructions.')
    print()
    print('Why Claude defaults to English without this:')
    print('  The surrounding context (system instructions + retrieved chunks) is mostly English.')
    print('  LLMs trained on text where response language matches surrounding language')
    print('  inherit this statistical bias. An explicit instruction cuts through it.')
    print()
    print('Model choice for multilingual RAG:')
    print('  claude-sonnet-4-6 handles all 5 demo languages natively.')
    print('  claude-haiku-4-5-20251001 is fine for simple retrieval-and-paraphrase tasks.')
    print('  For low-resource languages: test carefully; quality degrades with less training data.')


---
## Key Takeaways

1. **Embeddings are language-specific by default.** An English-only model hasn't learned that
   *reembolso* and *refund* are synonyms. They land in different vector neighborhoods because
   the model never saw them in similar contexts. This isn't a tuning problem — it's architectural.

2. **Three strategies, not one.** Translate to a pivot language (simple but lossy),
   use a multilingual embedding model (best for mixed workloads), or maintain per-language
   indexes with a router (best quality per language, highest operational cost).

3. **Translation is lossy in two ways you won't notice until it hurts.**  
   Wording drift (*refund* → *reimbursement*) splits one concept across multiple vector
   neighborhoods. Proper nouns (*Geschäftsführer*) become inconsistent English variants.
   The same German word, translated five times, can produce five different English words.

4. **The cross-lingual test is your go/no-go check.** Embed `"refund policy EN"` and
   `"política de reembolso ES"`. If their similarity is lower than the similarity between
   `"refund policy"` and `"how do I bake a cake"`, your model is not doing cross-lingual work.

5. **Tag every chunk with its language at index time.** Even with a multilingual model,
   the language tag unlocks filtering, preference ranking, per-language confidence scores,
   and regulatory compliance later. Adding it after the fact means re-indexing everything.

6. **Multilingual models handle code-switching gracefully.** Translation pipelines don't.
   For technical documentation that mixes English terms into other languages, a multilingual
   model is almost always the right call.

7. **Always pass detected language explicitly to the synthesis LLM.** Without
   `"Answer in Japanese. Do not switch to English."`, an LLM surrounded by English
   instructions and English chunks will answer in English — even when the user asked in Japanese.

8. **Multilingual amplifies every existing flaw.** Don't add it to a single-language
   RAG system that's still fragile. Get the monolingual version solid first, then extend.

---

*Up next — Lesson 9.5: My documents have tables, charts, and diagrams.*
*How do I RAG over content that isn't just text?*
